# E3 · The scaler refitted on the test set

**Type**: Find the bug · **Level**: ●●○ independent · **Time**: about 30 minutes · **core path ★**

## Objective

Find out why a model with excellent training results has a test error ten times larger, fix
the preprocessing and verify that a `Pipeline` produces exactly the same correct result.

## Dataset

`ds.trending_series()`: 400 chronological rows with two features `x1`, `x2` that **grow over time** and a target `y`.
The split is chronological: the first 300 rows for training, the last 100 for the test (you predict the future).

## Educational purpose

Remember what a scaler does: `fit` computes mean and standard deviation, `transform` applies them. If you call `fit` on
the test set too, you compute new ones and re-center the data, erasing exactly the information (the level reached
by the features) that the model had learned to use.

## Task

1. Run the colleague's cell: look at the test RMSE (`rmse_bug`), enormously higher than the training one.
2. Compute `test_mean_std_right`: the means of the two features of the test set standardized **with the statistics of the training set**. They are not zero: it is what the model should see.
3. Fix the preprocessing (`X_te_ok`) and compute `rmse_ok` with the same model.
4. Build `pipe = make_pipeline(StandardScaler(), LinearRegression())`, train it on `X_tr` and save the predictions in `pred_pipe`. They must match those of the correct manual preprocessing (`pred_manual`).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

d = ds.trending_series()
X, y = d[["x1", "x2"]], d["y"]
cut = int(0.75 * len(d))
X_tr, X_te, y_tr, y_te = X.iloc[:cut], X.iloc[cut:], y.iloc[:cut], y.iloc[cut:]

## Your work

### The colleague's code

In [ ]:
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.fit_transform(X_te)

model = LinearRegression().fit(X_tr_s, y_tr)
rmse_train = mean_squared_error(y_tr, model.predict(X_tr_s), squared=False)
rmse_bug = mean_squared_error(y_te, model.predict(X_te_s), squared=False)
print(f"RMSE train {rmse_train:.2f} | RMSE test {rmse_bug:.2f}")

### Your diagnosis and the fix

In [ ]:
test_mean_std_right = ...
print(test_mean_std_right)

In [ ]:
X_te_ok = ...
pred_manual = ...
rmse_ok = rmse_bug

In [ ]:
pipe = ...
pred_pipe = ...

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='The right method is `scaler.transform(X_te)`, without `fit`. `test_mean_std_right` is `scaler.transform(X_te).mean(axis=0)`. `pipe.fit(X_tr, y_tr)` fits the scaler on the training set alone and then applies `transform` (not `fit`) to the test.'):
    check(rmse_bug > 3 * rmse_ok, "the wrong preprocessing makes the error more than 3 times worse",
          "rmse_ok must come from X_te standardized with scaler.transform (training statistics)")
    check(np.all(np.abs(np.asarray(test_mean_std_right)) > 1),
          "in the test, the features are more than one standard deviation above the training level",
          "test_mean_std_right = scaler.transform(X_te).mean(axis=0): the trend takes the features far above the training means")
    check(close(pred_manual, model.predict(scaler.transform(X_te))), "pred_manual uses the scaler fitted on the training set",
          "pred_manual must be model.predict(scaler.transform(X_te))")
    check(close(pred_pipe, pred_manual), "the Pipeline produces exactly the same predictions",
          "the Pipeline trained on X_tr must give the same predictions as the correct manual preprocessing")
    check(rmse_ok < 2, "the fixed RMSE is close to the noise of the target", "rmse_ok should be about 1")
done()

---
The commented solution is in `solutions/E_find_the_bug/E3_scaler_refitted_on_test.ipynb`. Open it after you pass the check, or after you have really tried.